# 04 — SAC Teacher → SFT Dataset

**Phase 2→3 transition** · MSc Thesis — ECLIPSE project  
Supervisor: Dr. Panagiotis Kasnesis | Student: Antonios Bastoulis

---

Goal: produce a behavior-cloning dataset for fine-tuning a single 3-building SLM.

1. Train a SAC agent for `SAC_EPISODES` episodes on the full 6-building
   district (`BUILDINGS=[0..5]`). Per-building policies — `central_agent=False`.
2. Evaluate it (Phase I + Combined challenge scores).
3. Run the trained SAC deterministically for **one full year (8 760 steps)**
   on the same 6-building env, and dump `(state_text, action_tokens)` pairs as
   JSONL — but slice each step into TWO rows: one for `TRAINING_BUILDINGS=[0,1,2]`
   and one for `HELDOUT_BUILDINGS=[3,4,5]`. The student SLM learns a 3-building
   policy; at Phase 4 the same LoRA loads into both agents.
4. Ship the JSONL to Colab → notebook 05 does LoRA SFT on Gemma.

**Compute:** sections 0–4 (train + pickle the teacher) are the expensive part —
training is env-step-bound so a GPU helps only modestly; budget several hours
and consider running it on Colab. Sections 4b–5 (load pickle + dump JSONL) are
cheap (~30 s) and run anywhere.

## 0 — Config

All knobs in one place.
- `SAC_EPISODES` — teacher training budget. A bigger teacher is a higher SFT
  ceiling; 30 episodes produced a degenerate teacher (see § 2).
- `ACTION_SCALING` — caps the SAC action range. CityLearn defaults it to 0.5;
  we set 1.0 so the teacher can use the full `[-1, 1]` battery range (see § 2).

The env uses `MERLINReward` (the single project reward — see `src/env.py`).

All artefacts go under `notebooks/artifacts/` with a timestamp prefix so
retraining never overwrites a previous run.

In [ ]:
import sys, time, warnings, pickle, copy
from pathlib import Path

import numpy as np
import pandas as pd

# Make src/ importable
_ROOT = Path.cwd().parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

warnings.filterwarnings("ignore")

# ── Hyperparameters ────────────────────────────────────────────────────────
SAC_EPISODES   = 40           # teacher training budget — sets the SFT ceiling.
                               # 30 episodes gave a degenerate teacher (batteries
                               # empty ~59% of the year); see the § 2 note.
ACTION_SCALING = 1.0           # SAC action-range cap. CityLearn defaults this to
                               # 0.5, which makes the policy emit 0.5·tanh(·) ∈
                               # (-0.5, 0.5) — half the 11-token action vocabulary
                               # (CHARGE/DISCHARGE_60..100) is then unreachable.
                               # 1.0 = full [-1, 1] battery range. See § 2.
ARTIFACTS      = _ROOT / "notebooks" / "artifacts"
AGENT_DIR      = ARTIFACTS / "agents"
DATASET_DIR    = ARTIFACTS / "sft_datasets"
AGENT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

STAMP = time.strftime("%Y%m%d_%H%M%S")
print(f"Run stamp: {STAMP}")


## 1 — Imports

CityLearn's bundled `SAC` and `BasicBatteryRBC` (the RBC anchors the teacher quality check in § 3). Everything project-specific comes from `src/`: the env factory and `snapshot_state` from `src.env`, the eval helpers from `src.eval`, and the SFT-side helpers — `render_state`, `action_to_token`, `format_action_block`, `dump_sac_trajectory_jsonl`, and `make_sft_prompt` — from `src.sft`. The teacher trains on the full 6-building district; the dataset is sliced into two 3-building rows per step (§ 5).

In [2]:
import citylearn
from citylearn.agents.sac import SAC
from citylearn.agents.rbc import BasicBatteryRBC

from src.env  import (
    make_env, snapshot_state, SEED,
    BUILDINGS,           # full 6-building district (SAC teacher trains here)
    TRAINING_BUILDINGS,  # [0,1,2] — Phase 3 single-agent slice
    HELDOUT_BUILDINGS,   # [3,4,5] — second slice for 2x data + generalization test
)
from src.eval import evaluate, comparison_table
from src.sft  import (
    render_state, action_to_token, format_action_block,
    dump_sac_trajectory_jsonl, make_sft_prompt,
)

np.random.seed(SEED)
print(f"CityLearn {citylearn.__version__}")
print(f"SAC teacher trains on BUILDINGS={BUILDINGS} (6-bldg district)")
print(f"SFT dataset emits 3-bldg slices: {TRAINING_BUILDINGS} and {HELDOUT_BUILDINGS}")


CityLearn 2.6.0b2
SAC teacher trains on BUILDINGS=[0, 1, 2, 3, 4, 5] (6-bldg district)
SFT dataset emits 3-bldg slices: [0, 1, 2] and [3, 4, 5]


## 2 — Train SAC

`SAC_EPISODES` is the teacher budget and sets the SFT ceiling. The earlier
30-episode teacher was degenerate for distillation — it barely cycled the
batteries (empty ~59 % of the year) and its action trajectory collapsed onto a
handful of tokens. Train long here.

`ACTION_SCALING` is passed to SAC's `action_scaling_coefficienct` kwarg — note
CityLearn **misspells** it (extra `c`); the correctly-spelled name is silently
ignored. CityLearn defaults the coefficient to 0.5, so the policy emits
`0.5·tanh(·) ∈ (-0.5, 0.5)` and half the 11-token action vocabulary
(`CHARGE/DISCHARGE_60..100`) is unreachable by construction. We set 1.0 for the
full `[-1, 1]` range; the cell asserts the value actually took effect.

Training is env-step-bound (CPU-heavy); a GPU helps only modestly. Expect on the
order of minutes per episode — budget several hours for a long run, or run this
section on Colab and pickle the agent (§ 4).

In [ ]:
env_sac   = make_env(session_name=f"sac_{STAMP}")

# CityLearn's SAC constructor kwarg is MISSPELLED — 'action_scaling_coefficienct'
# (extra 'c', see citylearn/agents/rlc.py). The correctly-spelled name lands in
# **kwargs and is silently ignored, leaving the 0.5 default. The assert guards
# against that silent failure.
agent_sac = SAC(env=env_sac, seed=SEED,
                action_scaling_coefficienct=ACTION_SCALING)
assert agent_sac.action_scaling_coefficient == ACTION_SCALING, (
    f"action scaling not applied: got {agent_sac.action_scaling_coefficient}, "
    f"expected {ACTION_SCALING} — check the misspelled kwarg"
)
print(f"SAC action_scaling_coefficient = {agent_sac.action_scaling_coefficient}")

print(f"Training SAC — {SAC_EPISODES} episodes (last deterministic) …")
_t0 = time.time()
agent_sac.learn(episodes=SAC_EPISODES, deterministic_finish=True)
_elapsed = time.time() - _t0
print(f"SAC training: {_elapsed:.1f}s ({_elapsed / 60:.1f} min)")


## 3 — Evaluate SAC vs RBC

Two sanity numbers: a 1-episode RBC baseline plus the deterministic final SAC episode. If SAC's Phase I score isn't materially below RBC's, the teacher is too weak to distil from — raise `SAC_EPISODES` and rerun before touching the JSONL. The ZNE table is a secondary check (Nweye et al. 2024 protocol) showing whether the SAC is actually exploiting solar.

In [ ]:
# RBC reference (1 episode)
env_rbc   = make_env(session_name=f"rbc_{STAMP}")
agent_rbc = BasicBatteryRBC(env=env_rbc)
agent_rbc.learn(episodes=1)

res_rbc = evaluate(env_rbc, label="RBC")
res_sac = evaluate(env_sac, label=f"SAC ({SAC_EPISODES}ep)")

challenge_df, zne_df = comparison_table([res_rbc, res_sac])
print("Challenge scores (lower is better):")
display(challenge_df)
print("\nZNE:")
display(zne_df[["ZNE ratio (solar / import)", "self-consumption ratio"]])

## 4 — Save SAC Agent (inference-only pickle)

Persist the trained SAC so § 4b can re-dump JSONL without retraining (a long
teacher run is expensive). The replay buffers are cleared first — they aren't
needed for inference and bloat the pickle from a few MB to ~130 MB. Filename
includes a timestamp so multiple teacher runs can coexist.

In [ ]:
agent_inf = copy.deepcopy(agent_sac)
for buf in agent_inf.replay_buffer:
    buf.buffer.clear()

sac_path = AGENT_DIR / f"sac_merlin_inference_{STAMP}.pkl"
with open(sac_path, "wb") as f:
    pickle.dump(agent_inf, f)
print(f"SAC inference agent saved: {sac_path}  ({sac_path.stat().st_size / 1e6:.1f} MB)")

## 4b — Load SAC Agent and Dump JSONL (no retraining)

Use this if you already have a saved inference agent in notebooks/artifacts/agents and you want to generate a new JSONL dataset without retraining.

In [ ]:
# Pick the most recent inference agent
candidates = sorted(AGENT_DIR.glob("sac_merlin_inference_*.pkl"))
if not candidates:
    raise FileNotFoundError(
        f"No inference agents found in {AGENT_DIR}. Run section 4 or add a .pkl first."
    )

sac_path = candidates[-1]
with open(sac_path, "rb") as f:
    agent_sac_loaded = pickle.load(f)
print(f"Loaded SAC inference agent: {sac_path}")

# Fresh env for the dump (resets to t=0)
env_dump = make_env(buildings=BUILDINGS)

jsonl_path = DATASET_DIR / f"sac_merlin_distill_{STAMP}_loaded.jsonl"

_t0 = time.time()
stats = dump_sac_trajectory_jsonl(
    env=env_dump,
    agent=agent_sac_loaded,
    out_path=jsonl_path,
    snapshot_fn=snapshot_state,
    building_slices=[TRAINING_BUILDINGS, HELDOUT_BUILDINGS],
    seed=SEED,
)
print(f"Dumped {stats['n_steps']} env steps × {stats['n_slices']} slices "
      f"= {stats['n_rows']} rows to {stats['path']}")
print(f"  per-row buildings = {stats['n_buildings']}  |  elapsed {time.time() - _t0:.1f}s")

## 5 — Dump 1-Year Trajectory as SFT JSONL (3-bldg slices)

We roll the trained SAC out for one full episode (8 760 hourly steps) on the
full 6-building district. At every step we emit **two** JSONL rows — one for
slice `[0,1,2]` and one for `[3,4,5]` — by selecting the relevant buildings
from the snapshot and the corresponding SAC actions.

**Why slice instead of training SAC on 3 buildings?**
SAC is per-building (`central_agent=False`), so each building's policy is
independent — slicing introduces zero distribution mismatch and saves a Colab
session. **Why two slices?** The SLM learns a building-agnostic 3-building
policy: at Phase 4 deployment the same LoRA loads into agent α (sees B0–2)
and agent β (sees B3–5) without retraining.

Each row:
- **prompt**: `STATE:\n` + `render_state(snapshot[slice])` — exactly what the
  SLM sees at inference time on its 3-building slice.
- **response**: 3 `<action building=i>TOKEN</action>` lines (i ∈ {0,1,2} —
  always local indices in the slice, not global).
- **slice**: the global building indices this row was emitted for.


In [ ]:
# Fresh env for the dump (resets to t=0) — SAC needs the full 6-building env
# it was trained on; slicing happens at JSONL emission time.
# No session_name → no render output. The dump is a JSONL-only artifact, the
# per-step CSV files render mode would write are wasted I/O on the MacBook.
env_dump = make_env(buildings=BUILDINGS)

jsonl_path = DATASET_DIR / f"sac_merlin_distill_{STAMP}.jsonl"

_t0 = time.time()
stats = dump_sac_trajectory_jsonl(
    env=env_dump,
    agent=agent_sac,
    out_path=jsonl_path,
    snapshot_fn=snapshot_state,
    building_slices=[TRAINING_BUILDINGS, HELDOUT_BUILDINGS],
    seed=SEED,                       # deterministic reset across re-runs
)
print(f"Dumped {stats['n_steps']} env steps × {stats['n_slices']} slices "
      f"= {stats['n_rows']} rows to {stats['path']}")
print(f"  per-row buildings = {stats['n_buildings']}  |  elapsed {time.time() - _t0:.1f}s")


## 6 — Inspect Dataset (sanity check)

Cheap checks before pushing the JSONL to Colab:

1. **Row count** should equal `n_steps × n_slices` — for a full-year 2-slice dump that's `8759 × 2 = 17518`.
2. **Two sample rows from different slices** — confirms the slice metadata is correct and the response always uses local building indices (0,1,2), not global.
3. **Action-token distribution** — the SAC teacher is heavily biased toward `DISCHARGE_20` / `IDLE` / `CHARGE_20`. That is expected (small actions dominate the optimal policy), but a degenerate dump (e.g. 99 % IDLE) would be visible here before wasting Colab time.

In [8]:
import json

lines = jsonl_path.read_text().strip().split("\n")
print(f"Total examples: {len(lines)}  (expect ~2× n_steps when using 2 slices)")

# Show one row from each slice near mid-year
mid = len(lines) // 2
for offset in (0, 1):
    row = json.loads(lines[mid + offset])
    print(f"\n── Sample row {mid+offset} (slice={row.get('slice')}, t={row.get('t')}) ──")
    print("PROMPT:")
    print(row["prompt"])
    print("\nRESPONSE:")
    print(row["response"])

# Action token distribution — sanity on bucket spread
from collections import Counter
import re
tok_re = re.compile(r"<action building=\d+>([A-Z_0-9]+)</action>")
all_toks = []
for ln in lines:
    all_toks.extend(tok_re.findall(json.loads(ln)["response"]))
print(f"\nAction-token distribution (top 12 of {len(all_toks)} tokens):")
for tok, n in Counter(all_toks).most_common(12):
    print(f"  {tok:14s} {n:6d}")


Total examples: 17518  (expect ~2× n_steps when using 2 slices)

── Sample row 8759 (slice=[3, 4, 5], t=4379) ──
PROMPT:
STATE:
Month 1, Mon 10:00  |  price=0.210 (LOW)  |  carbon=0.178 (MID)
Buildings:
  B0: SoC=  0.0%  load=2.07 kWh  last_net=-0.13 kWh  solar=HIGH
  B1: SoC= 16.0%  load=0.84 kWh  last_net=+0.70 kWh  solar=HIGH
  B2: SoC= 99.7%  load=0.00 kWh  last_net=-2.75 kWh  solar=HIGH

RESPONSE:
<action building=0>IDLE</action>
<action building=1>CHARGE_60</action>
<action building=2>IDLE</action>

── Sample row 8760 (slice=[0, 1, 2], t=4380) ──
PROMPT:
STATE:
Month 1, Mon 11:00  |  price=0.210 (LOW)  |  carbon=0.171 (MID)
Buildings:
  B0: SoC= 60.5%  load=0.32 kWh  last_net=-0.03 kWh  solar=HIGH
  B1: SoC= 52.8%  load=0.32 kWh  last_net=+0.05 kWh  solar=HIGH
  B2: SoC= 99.7%  load=0.16 kWh  last_net=-1.77 kWh  solar=HIGH

RESPONSE:
<action building=0>CHARGE_60</action>
<action building=1>CHARGE_40</action>
<action building=2>IDLE</action>

Action-token distribution (top 12 of 5

## 7 — Show the SFT prompt template (used in notebook 05)

Notebook 05 will wrap each row as:

```
<system> make_sft_prompt(3) </system>
<user>   STATE:\n... </user>            ← row["prompt"]
<assistant> <action building=0>...   </assistant>  ← row["response"]
```

Each row covers a 3-building slice — local building indices 0–2 inside the
prompt/response always refer to positions within the slice, not global B0–B5.


In [8]:
print(make_sft_prompt(3))

You are an energy management agent for 3 buildings. Goal: minimize grid dependency and energy costs over time.

[Actions] — choose exactly one per building:
CHARGE_100, CHARGE_80, CHARGE_60, CHARGE_40, CHARGE_20, IDLE, DISCHARGE_20, DISCHARGE_40, DISCHARGE_60, DISCHARGE_80, DISCHARGE_100

[State Variables & Environment]
- 'price': Current cost of grid electricity. PEAK indicates high cost.
- 'solar': Renewable energy generated locally.
- 'load': Energy demanded by the building's operations. High load means the building needs a lot of power.
- 'SoC': Battery State of Charge (0% = empty, 100% = full).
- Charging stores energy. Doing so when solar is HIGH or price is LOW is efficient, but charging from the grid increases district demand.
- Discharging uses stored energy to serve the 'load', directly reducing grid dependency. This is highly beneficial when 'price' is PEAK or 'load' is high and SoC is sufficient.
- No forecasts are provided. Plan ahead by predicting how price, solar, and lo

## 8 — Hand-off to Colab

**Commit and push** the JSONL artifact (it's a few MB) so notebook 05
can `git pull` it in Colab. Or upload manually via the Files panel.

```bash
git add notebooks/artifacts/sft_datasets/sac_*.jsonl
git commit -m "sac distill dataset (full-year teacher rollout)"
git push
```

Then open `notebooks/05_sft_gemma_colab.ipynb` in Colab.